# LSTM T2D Risk Prediction -- Training + Evaluation (Masked, BERT-parity)

**Antes de correr nada**, sube estos 3 archivos a la MISMA carpeta de tu Google Drive:
- `matched_test.csv`
- `EARLIEST_DX_deid.csv`
- `regular_patients_1_year.pkl`

y pon la ruta de esa carpeta en `DRIVE_FOLDER` en la siguiente celda. Todas las demás rutas del notebook son solo el nombre del archivo -- ya no hay rutas largas de Windows.

Corre las celdas en orden, de arriba a abajo. Entrena 10 iteraciones independientes (para el análisis de estabilidad), evalúa en el test interno balanceado y en el externo balanceado (igual que el BERT), y guarda todo: métricas por iteración, resumen agregado, confusion matrices, y el mejor modelo (`.pth`).

**Guardado automático por época**: si Colab se desconecta a mitad de entrenamiento, simplemente vuelve a correr el notebook desde el principio -- detecta el checkpoint y retoma justo donde se quedó, sin perder progreso ni duplicar resultados.


**Nota sobre pandas**: la celda de instalación actualiza pandas (necesario para poder leer los .pkl generados en tu máquina). La primera vez que la corras, verás un aviso pidiéndote que hagas *Runtime > Restart session* y luego *Runtime > Run all* desde el principio -- eso es normal y solo hace falta una vez por sesión de Colab.


In [ ]:
# %% Mount Google Drive and set the working folder
from google.colab import drive
drive.mount('/content/drive')

import os

# <<< EDIT THIS to the Drive folder where you uploaded the 3 files >>>
DRIVE_FOLDER = "/content/drive/MyDrive/EHR-LSTM"

os.chdir(DRIVE_FOLDER)
print("Working directory:", os.getcwd())
print("Files found here:", [f for f in os.listdir('.') if f.endswith(('.csv', '.pkl'))])


In [ ]:
# %% Install / upgrade packages
# pandas MUST be upgraded here -- the .pkl files were created locally with a
# recent pandas (2.x) that stores dates as datetime64[us]. Colab ships an
# older pandas by default that can't deserialize that dtype and raises
# NotImplementedError when loading the pickle otherwise.
!pip install -q --upgrade pandas
!pip install -q imbalanced-learn

import pandas as pd
print('pandas version:', pd.__version__)
print('If this is the FIRST time running this cell in this session: go to')
print('Runtime > Restart session now, then Runtime > Run all (from the top,')
print('including the Drive mount cell) -- a plain pip upgrade does not take')
print('effect in an already-running Python session.')


In [ ]:
# %% Imports
import os
import pickle
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              confusion_matrix, ConfusionMatrixDisplay, roc_auc_score)
from sklearn.model_selection import StratifiedShuffleSplit
from imblearn.under_sampling import RandomUnderSampler
import matplotlib.pyplot as plt
import tqdm

print("Imports OK.")


In [ ]:
# %% Streaming pickle load/save (inlined -- no separate file needed in Colab)
# Reads either a single-object pickle (old style) or the incremental
# (patient_count, then one (pid, df) pair per pickle.dump call) format used
# by the rest of the preprocessing pipeline -- auto-detected.
def load_streamed_patient_dict(path, desc="Loading"):
    with open(path, "rb") as f:
        first_obj = pickle.load(f)
        if isinstance(first_obj, dict):
            return first_obj
        n_patients = first_obj
        patient_data = {}
        for _ in tqdm.tqdm(range(n_patients), desc=desc, colour='blue'):
            pid, df = pickle.load(f)
            patient_data[pid] = df
        return patient_data


In [ ]:
# %% Configuration & Hyperparameters
class Config:
    # Simple filenames -- we already cd'd into DRIVE_FOLDER above, so these
    # just need to be the 3 files you uploaded there.
    patient_data_path = "regular_patients_1_year.pkl"
    earliest_dx_path = "EARLIEST_DX_deid.csv"
    external_test_path = "matched_test.csv"

    # BERT parity: confirmed architecture uses only age (word/position/
    # token_type/age embeddings -- no gender/race/ethnicity in any form).
    DEMO_COLUMNS = ['AGE_AT_END']

    # Architecture
    LSTM_HIDDEN_SIZE = 128
    DEMO_HIDDEN_SIZE = 32
    COMBINED_HIDDEN_SIZE = 64
    NUM_LAYERS = 2
    DROPOUT = 0.5

    # Training
    BATCH_SIZE = 128
    EPOCHS = 1000
    LEARNING_RATE = 0.5 * 1e-4
    WEIGHT_DECAY = 1e-5
    PATIENCE = 15
    MIN_DELTA = 1e-4
    TEST_SPLIT_SIZE = 0.3
    RANDOM_STATE = 42

    # Outputs -- all saved directly in DRIVE_FOLDER/models/, so they persist
    # in your Drive even if the Colab runtime disconnects/recycles.
    # NOTE: trained on the "regular_patients_1_year" visit-CADENCE subset,
    # which is NOT the same as a "1-year horizon" model (see the decisions
    # log) -- named accordingly so it's not confused with future horizon work.
    MODEL_OUTPUT_DIR = "models"
    MODEL_NAME = "lstm_model_regular_1year_best.pth"
    CHECKPOINT_PATH = "models/training_checkpoint.pth"  # autosaved every epoch; resumes automatically if found

config = Config()
os.makedirs(config.MODEL_OUTPUT_DIR, exist_ok=True)

# Use GPU if available.
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f" Using device: {DEVICE}" + (f" ({torch.cuda.get_device_name(0)})" if DEVICE.type == 'cuda' else " -- no GPU detected, training will be slower"))


In [ ]:
# %% Load Data
imputed_patients_matrices_all = load_streamed_patient_dict(config.patient_data_path, desc="Loading training data")
df2 = pd.read_csv(config.earliest_dx_path)
print(f" Loaded {len(imputed_patients_matrices_all)} patients and the diagnosis-date file.")

# --- Process Diagnosis Dates -- MINIMUM necessary to avoid the leakage bug ---
# EARLIEST_DX_deid.csv can have MULTIPLE rows per patient (one per diagnosis
# code). Using it as-is (or joining without dedup) is exactly the bug that
# hit the BERT pipeline earlier in this project. The only safe operation
# here is: sort by date ascending, then keep the FIRST (=earliest) row per
# PATIENT_ID. Nothing else should be done to this dataframe.
df2 = df2.drop(columns=['DX'], errors='ignore')  # drop anything that isn't the date, so it can't accidentally get used
df2['EARLIEST_DX'] = pd.to_datetime(df2['EARLIEST_DX'], errors='coerce')
df2_earliest = df2.sort_values('EARLIEST_DX').drop_duplicates(subset='PATIENT_ID', keep='first')

# Safety check -- if this ever fails, STOP and investigate before training;
# it means the leakage-prevention dedup above did not work as intended.
assert df2_earliest['PATIENT_ID'].is_unique, \
    "EARLIEST_DX_deid.csv still has duplicate PATIENT_IDs after dedup -- DO NOT PROCEED, this is the leakage bug."
print(f" Dedup check passed: {len(df2_earliest)} unique patients with a diagnosis date "
      f"(from {len(df2)} raw rows in the file).")

imputed_patients = set(imputed_patients_matrices_all.keys())
df2_filtered = df2_earliest[df2_earliest['PATIENT_ID'].isin(imputed_patients)]
dx_date_dict = df2_filtered.set_index('PATIENT_ID')['EARLIEST_DX'].dt.date.to_dict()
print(f" {len(dx_date_dict)} patients in this cohort have a matched diagnosis date.")


In [ ]:
# %% Load External Test Patient IDs
# These patients are drawn from the SAME underlying cohort (confirmed with
# the thesis advisor) -- so they MUST be excluded from the internal
# train/val/test pool below, or "external" validation would partly be
# evaluating on data the model was trained on, which defeats its purpose.
try:
    ext_df = pd.read_csv(config.external_test_path)
    external_test_ids = set(ext_df['PATIENT_ID'].astype(str).unique())
    print(f" External test set: {len(external_test_ids)} patient IDs loaded "
          f"(excluded from internal train/val/test).")
except Exception as e:
    print(f" Could not load external test IDs ({e}) -- external evaluation will be skipped.")
    external_test_ids = set()


In [ ]:
# %% Prepare Visit-Level Sequences (masked) + Labels
print("Extracting demographics by title. Remaining rows are sequential...")

X_sequential, X_demographics, y_patient_labels = [], [], []
pids_per_sample = []
X_ext_sequential, X_ext_demographics, y_ext_labels = [], [], []
n_recovered_positives = 0  # diagnostic: patients rescued by the mislabeling fix below

first_pid = next(iter(imputed_patients_matrices_all))
all_indices = imputed_patients_matrices_all[first_pid].index
seq_indices = [idx for idx in all_indices if idx not in config.DEMO_COLUMNS]

config.ORIGINAL_SEQ_SIZE = len(seq_indices)
config.SEQ_INPUT_SIZE = config.ORIGINAL_SEQ_SIZE * 2
config.DEMO_INPUT_SIZE = len(config.DEMO_COLUMNS)

print(f" Features detected: {len(seq_indices)} sequential, {len(config.DEMO_COLUMNS)} demographic.")

for pid, df_patient in tqdm.tqdm(imputed_patients_matrices_all.items(), desc="Processing Patients", colour='cyan'):
    demo_features = df_patient.loc[config.DEMO_COLUMNS].iloc[:, 0].values
    seq_raw = df_patient.loc[seq_indices].T.values

    mask = (~np.isnan(seq_raw)).astype(np.float32)
    seq_vals = np.nan_to_num(seq_raw, nan=0.0)
    masked_features = np.concatenate([seq_vals, mask], axis=1)

    visit_dates = [pd.to_datetime(col).date() for col in df_patient.columns]
    diagnosis_date = dx_date_dict.get(pid)
    first_dx_idx = -1
    if diagnosis_date:
        for i, v_date in enumerate(visit_dates):
            if v_date >= diagnosis_date:
                first_dx_idx = i
                break
        else:
            # If NO visit reaches the diagnosis date, every captured visit is
            # legitimately pre-diagnosis -- this patient should be POSITIVE
            # using all of them, not silently mislabeled as a negative
            # control (which is what happens if first_dx_idx is left at -1).
            first_dx_idx = len(visit_dates)
            n_recovered_positives += 1

    if first_dx_idx != -1:
        if first_dx_idx >= 2:
            is_external = str(pid) in external_test_ids
            target_seq = X_ext_sequential if is_external else X_sequential
            target_demo = X_ext_demographics if is_external else X_demographics
            target_label = y_ext_labels if is_external else y_patient_labels
            target_seq.append(masked_features[:first_dx_idx, :])
            target_demo.append(demo_features)
            target_label.append(1)
            if not is_external:
                pids_per_sample.append(pid)
    else:
        # Full visit history for non-T2D patients (no trimming of the last
        # visit) -- matches the BERT pipeline, which doesn't drop visits for
        # negative patients either.
        if df_patient.shape[1] >= 2:
            is_external = str(pid) in external_test_ids
            target_seq = X_ext_sequential if is_external else X_sequential
            target_demo = X_ext_demographics if is_external else X_demographics
            target_label = y_ext_labels if is_external else y_patient_labels
            target_seq.append(masked_features)
            target_demo.append(demo_features)
            target_label.append(0)
            if not is_external:
                pids_per_sample.append(pid)


In [ ]:
# %% Undersampling and Splitting (internal set)
print(f"\n{n_recovered_positives} confirmed-T2D patients had no captured visit reaching their "
      f"diagnosis date -- correctly labeled positive using their full pre-diagnosis visit history.")
print(f"Final label counts before balancing: {sum(1 for y in y_patient_labels if y == 1)} positive, "
      f"{sum(1 for y in y_patient_labels if y == 0)} negative (internal pool).")
print(f"External test pool (excluded from internal pool): "
      f"{sum(1 for y in y_ext_labels if y == 1)} positive, {sum(1 for y in y_ext_labels if y == 0)} negative.")

rus = RandomUnderSampler(sampling_strategy=1.0, random_state=config.RANDOM_STATE)
all_indices = np.arange(len(y_patient_labels)).reshape(-1, 1)
resampled_indices_flat, y_balanced_raw = rus.fit_resample(all_indices, y_patient_labels)
resampled_indices = resampled_indices_flat.flatten()

y_balanced = y_balanced_raw if isinstance(y_balanced_raw, list) else y_balanced_raw.tolist()
X_seq_balanced = [X_sequential[i] for i in resampled_indices]
X_demo_balanced = [X_demographics[i] for i in resampled_indices]

sss = StratifiedShuffleSplit(n_splits=1, test_size=config.TEST_SPLIT_SIZE, random_state=config.RANDOM_STATE)
train_idx, temp_idx = next(sss.split(X_demo_balanced, y_balanced))
val_idx = temp_idx[:len(temp_idx) // 2]
test_idx = temp_idx[len(temp_idx) // 2:]

def get_data(idxs):
    return [X_seq_balanced[i] for i in idxs], [X_demo_balanced[i] for i in idxs], [y_balanced[i] for i in idxs]

X_train_seq, X_train_demo, y_train = get_data(train_idx)
X_val_seq, X_val_demo, y_val = get_data(val_idx)
X_test_seq, X_test_demo, y_test = get_data(test_idx)
print(f"Internal split -- train: {len(y_train)}, val: {len(y_val)}, test: {len(y_test)}")


In [ ]:
# %% PyTorch Dataset and DataLoader
class PatientDataset(Dataset):
    def __init__(self, seq_data, demo_data, labels):
        self.seq_data = seq_data
        self.demo_data = demo_data
        self.labels = labels
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx): return self.seq_data[idx], self.demo_data[idx], self.labels[idx]

def collate_fn(batch):
    seq_batch, demo_batch, labels_batch = zip(*batch)
    seq_tensors = [torch.tensor(x, dtype=torch.float32) for x in seq_batch]
    lengths = torch.tensor([len(x) for x in seq_tensors])
    seq_padded = pad_sequence(seq_tensors, batch_first=True, padding_value=0.0)
    demo_tensor = torch.tensor(np.array(demo_batch), dtype=torch.float32)
    labels_tensor = torch.tensor(labels_batch, dtype=torch.float32)
    return seq_padded, demo_tensor, labels_tensor, lengths

train_loader = DataLoader(PatientDataset(X_train_seq, X_train_demo, y_train), batch_size=config.BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(PatientDataset(X_val_seq, X_val_demo, y_val), batch_size=config.BATCH_SIZE, collate_fn=collate_fn)
test_loader = DataLoader(PatientDataset(X_test_seq, X_test_demo, y_test), batch_size=config.BATCH_SIZE, collate_fn=collate_fn)


In [ ]:
# %% External Test Set -- balanced 1:1, same as internal (per thesis advisor's
# instruction: BERT's reported results/confusion matrix were also produced
# on a balanced/subsampled set, so this matches that methodology).
ext_loader = None
if y_ext_labels:
    ext_indices = np.arange(len(y_ext_labels)).reshape(-1, 1)
    rus_ext = RandomUnderSampler(sampling_strategy=1.0, random_state=config.RANDOM_STATE)
    ext_resampled_idx_flat, y_ext_balanced = rus_ext.fit_resample(ext_indices, y_ext_labels)
    ext_resampled_idx = ext_resampled_idx_flat.flatten()
    X_ext_seq_balanced = [X_ext_sequential[i] for i in ext_resampled_idx]
    X_ext_demo_balanced = [X_ext_demographics[i] for i in ext_resampled_idx]
    y_ext_balanced = y_ext_balanced if isinstance(y_ext_balanced, list) else y_ext_balanced.tolist()
    print(f" External test set balanced via undersampling: {len(y_ext_labels)} -> {len(y_ext_balanced)} patients "
          f"({sum(1 for y in y_ext_balanced if y==1)} positive, {sum(1 for y in y_ext_balanced if y==0)} negative).")
    ext_loader = DataLoader(PatientDataset(X_ext_seq_balanced, X_ext_demo_balanced, y_ext_balanced),
                             batch_size=config.BATCH_SIZE, collate_fn=collate_fn)
else:
    print(" No external test patients found -- external evaluation will be skipped.")


In [ ]:
# %% Model Definition
class PatientRiskModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=config.SEQ_INPUT_SIZE,
            hidden_size=config.LSTM_HIDDEN_SIZE,
            num_layers=config.NUM_LAYERS,
            batch_first=True,
            dropout=config.DROPOUT if config.NUM_LAYERS > 1 else 0.0
        )
        self.demo_encoder = nn.Sequential(
            nn.Linear(config.DEMO_INPUT_SIZE, config.DEMO_HIDDEN_SIZE),
            nn.ReLU(),
            nn.Dropout(config.DROPOUT)
        )
        self.classifier = nn.Sequential(
            nn.Linear(config.LSTM_HIDDEN_SIZE + config.DEMO_HIDDEN_SIZE, config.COMBINED_HIDDEN_SIZE),
            nn.ReLU(),
            nn.Dropout(config.DROPOUT),
            nn.Linear(config.COMBINED_HIDDEN_SIZE, 1)
        )

    def forward(self, x_seq, x_demo, lengths):
        packed_x = pack_padded_sequence(x_seq, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (h_n, _) = self.lstm(packed_x)
        seq_embedding = h_n[-1]
        demo_embedding = self.demo_encoder(x_demo)
        combined_embedding = torch.cat((seq_embedding, demo_embedding), dim=1)
        return self.classifier(combined_embedding).squeeze(-1)


In [ ]:
# %% Training and Evaluation Functions (with per-epoch autosave)

def save_checkpoint(iteration_num, epoch, model_state_dict, optimizer_state_dict,
                     best_val_loss, patience_counter, best_model_state_iter):
    """Autosaved after every epoch. References the notebook-global result
    accumulators (all_metrics, etc.) directly -- only ever READ here, and
    guaranteed to exist by the time this is called from inside the training loop."""
    torch.save({
        'iteration': iteration_num,
        'epoch': epoch,
        'model_state_dict': model_state_dict,
        'optimizer_state_dict': optimizer_state_dict,
        'best_val_loss': best_val_loss,
        'patience_counter': patience_counter,
        'best_model_state_iter': best_model_state_iter,
        'all_metrics': all_metrics,
        'external_metrics': external_metrics,
        'per_iteration_records': per_iteration_records,
        'best_test_auc': best_test_auc,
        'best_model_state_overall': best_model_state,
        'best_iter_num': best_iter_num,
        'best_cm_internal': best_cm_internal,
        'best_cm_external': best_cm_external,
    }, config.CHECKPOINT_PATH)


def train_model(model, train_loader, val_loader, config, iteration_num,
                 resume_epoch=0, resume_optimizer_state=None,
                 resume_best_val_loss=float('inf'), resume_patience_counter=0,
                 resume_best_model_state_iter=None):
    model = model.to(DEVICE)
    train_labels = np.array(train_loader.dataset.labels)
    pos_weight = torch.tensor(np.sum(train_labels == 0) / np.sum(train_labels == 1), dtype=torch.float32).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = optim.Adam(model.parameters(), lr=config.LEARNING_RATE, weight_decay=config.WEIGHT_DECAY)
    if resume_optimizer_state is not None:
        optimizer.load_state_dict(resume_optimizer_state)

    best_val_loss, patience_counter = resume_best_val_loss, resume_patience_counter
    best_model_state_iter = resume_best_model_state_iter
    if resume_epoch > 0:
        print(f" Resuming iteration {iteration_num+1} from epoch {resume_epoch+1} "
              f"(best_val_loss so far: {best_val_loss:.4f}, patience: {patience_counter}/{config.PATIENCE})")

    for epoch in range(resume_epoch, config.EPOCHS):
        model.train()
        total_loss = 0
        for seq_batch, demo_batch, y_batch, lengths in train_loader:
            seq_batch, demo_batch, y_batch = seq_batch.to(DEVICE), demo_batch.to(DEVICE), y_batch.to(DEVICE)
            optimizer.zero_grad()
            logits = model(seq_batch, demo_batch, lengths)
            loss = criterion(logits, y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item()

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for seq_batch, demo_batch, y_batch, lengths in val_loader:
                seq_batch, demo_batch, y_batch = seq_batch.to(DEVICE), demo_batch.to(DEVICE), y_batch.to(DEVICE)
                logits = model(seq_batch, demo_batch, lengths)
                val_loss += criterion(logits, y_batch).item()

        val_loss /= len(val_loader)
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1:03} | Train Loss: {total_loss/len(train_loader):.4f} | Val Loss: {val_loss:.4f}")

        if best_val_loss - val_loss > config.MIN_DELTA:
            best_val_loss, patience_counter = val_loss, 0
            best_model_state_iter = {k: v.cpu() for k, v in model.state_dict().items()}
        else:
            patience_counter += 1

        # Autosave after EVERY epoch -- if the Colab runtime disconnects,
        # re-running the notebook picks this back up automatically.
        save_checkpoint(
            iteration_num, epoch + 1,
            {k: v.cpu() for k, v in model.state_dict().items()},
            optimizer.state_dict(),
            best_val_loss, patience_counter, best_model_state_iter,
        )

        if patience_counter >= config.PATIENCE:
            print(f"\u23f3 Early stopping at epoch {epoch+1}")
            break

    if best_model_state_iter: model.load_state_dict(best_model_state_iter)
    return model


def evaluate(model, data_loader, title="Evaluation"):
    model = model.to(DEVICE)
    model.eval()
    y_true, y_prob = [], []
    with torch.no_grad():
        for seq_batch, demo_batch, y_batch, lengths in data_loader:
            seq_batch, demo_batch = seq_batch.to(DEVICE), demo_batch.to(DEVICE)
            logits = model(seq_batch, demo_batch, lengths)
            y_true.extend(y_batch.tolist())
            y_prob.extend(torch.sigmoid(logits).tolist())

    y_pred = [1 if p > 0.5 else 0 for p in y_prob]

    acc = accuracy_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_prob)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)  # sensitivity / TPR
    f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
    cm = confusion_matrix(y_true, y_pred)
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    else:
        specificity = 0.0

    print(f"\n --- {title} ---")
    print(f"ACC  : {acc:.4f}")
    print(f"AUC  : {auc:.4f}")
    print(f"PREC : {prec:.4f}")
    print(f"REC  : {rec:.4f}  (sensitivity)")
    print(f"SPEC : {specificity:.4f}  (specificity)")
    print(f"F1   : {f1_macro:.4f}")
    print(f"Confusion matrix [[TN FP] [FN TP]]:\n{cm}")

    return acc, auc, prec, rec, f1_macro, cm, specificity


In [ ]:
# %% Resume support: load a previous checkpoint if one exists
NUM_ITERATIONS = 10

start_iteration = 0
resume_epoch = 0
resume_model_state = None
resume_optimizer_state = None
resume_best_val_loss = float('inf')
resume_patience_counter = 0
resume_best_model_state_iter = None

if os.path.exists(config.CHECKPOINT_PATH):
    print(f" Found existing checkpoint at {config.CHECKPOINT_PATH} -- resuming...")
    ckpt = torch.load(config.CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
    start_iteration = ckpt['iteration']
    resume_epoch = ckpt['epoch']
    resume_model_state = ckpt['model_state_dict']
    resume_optimizer_state = ckpt['optimizer_state_dict']
    resume_best_val_loss = ckpt['best_val_loss']
    resume_patience_counter = ckpt['patience_counter']
    resume_best_model_state_iter = ckpt['best_model_state_iter']
    all_metrics = ckpt['all_metrics']
    external_metrics = ckpt['external_metrics']
    per_iteration_records = ckpt['per_iteration_records']
    best_test_auc = ckpt['best_test_auc']
    best_model_state = ckpt['best_model_state_overall']
    best_iter_num = ckpt['best_iter_num']
    best_cm_internal = ckpt['best_cm_internal']
    best_cm_external = ckpt['best_cm_external']
    print(f" Resuming at iteration {start_iteration+1}/{NUM_ITERATIONS}, epoch {resume_epoch+1} "
          f"({len(per_iteration_records)} result rows already completed).")
else:
    all_metrics = {'acc': [], 'auc': [], 'prec': [], 'rec': [], 'spec': [], 'f1': []}
    external_metrics = {'acc': [], 'auc': [], 'prec': [], 'rec': [], 'spec': [], 'f1': []}
    best_test_auc = -1.0
    best_model_state = None
    best_iter_num = None
    best_cm_internal = None
    best_cm_external = None
    per_iteration_records = []

print(f" Starting from iteration {start_iteration+1}/{NUM_ITERATIONS} for stability analysis...")


In [ ]:
# %% 10 Iterations for Stability Analysis (this is the long-running cell)
for i in tqdm.tqdm(range(start_iteration, NUM_ITERATIONS), desc="Mask Training Iterations", colour='green'):
    print("\n" + "=" * 40)
    print(f" ITERATION {i+1}/{NUM_ITERATIONS} ")
    print("=" * 40)

    # 1. New dynamic split per iteration (deterministic by seed, so resuming
    #    the same iteration reproduces the exact same split).
    current_seed = config.RANDOM_STATE + i
    sss_iter = StratifiedShuffleSplit(n_splits=1, test_size=config.TEST_SPLIT_SIZE, random_state=current_seed)
    train_idx, temp_idx = next(sss_iter.split(X_demo_balanced, y_balanced))
    val_idx = temp_idx[:len(temp_idx) // 2]
    test_idx = temp_idx[len(temp_idx) // 2:]

    X_tr_s, X_tr_d, y_tr = get_data(train_idx)
    X_va_s, X_va_d, y_va = get_data(val_idx)
    X_te_s, X_te_d, y_te = get_data(test_idx)

    train_loader_iter = DataLoader(PatientDataset(X_tr_s, X_tr_d, y_tr), batch_size=config.BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
    val_loader_iter = DataLoader(PatientDataset(X_va_s, X_va_d, y_va), batch_size=config.BATCH_SIZE, collate_fn=collate_fn)
    test_loader_iter = DataLoader(PatientDataset(X_te_s, X_te_d, y_te), batch_size=config.BATCH_SIZE, collate_fn=collate_fn)

    # 3. Model reset (or resume) and training
    is_resumed_iteration = (i == start_iteration and resume_epoch > 0)
    model_iter = PatientRiskModel(config)
    if is_resumed_iteration and resume_model_state is not None:
        model_iter.load_state_dict(resume_model_state)

    trained_model_iter = train_model(
        model_iter, train_loader_iter, val_loader_iter, config, iteration_num=i,
        resume_epoch=resume_epoch if is_resumed_iteration else 0,
        resume_optimizer_state=resume_optimizer_state if is_resumed_iteration else None,
        resume_best_val_loss=resume_best_val_loss if is_resumed_iteration else float('inf'),
        resume_patience_counter=resume_patience_counter if is_resumed_iteration else 0,
        resume_best_model_state_iter=resume_best_model_state_iter if is_resumed_iteration else None,
    )

    # 4. Internal test evaluation
    acc, auc, prec, rec, f1, cm_internal, spec = evaluate(trained_model_iter, test_loader_iter, f"Test Set Iter {i+1}")
    all_metrics['acc'].append(acc)
    all_metrics['auc'].append(auc)
    all_metrics['prec'].append(prec)
    all_metrics['rec'].append(rec)
    all_metrics['spec'].append(spec)
    all_metrics['f1'].append(f1)

    # 5. External test evaluation
    cm_external = None
    if ext_loader is not None:
        e_acc, e_auc, e_prec, e_rec, e_f1, cm_external, e_spec = evaluate(trained_model_iter, ext_loader, f"External Test Iter {i+1}")
        external_metrics['acc'].append(e_acc)
        external_metrics['auc'].append(e_auc)
        external_metrics['prec'].append(e_prec)
        external_metrics['rec'].append(e_rec)
        external_metrics['spec'].append(e_spec)
        external_metrics['f1'].append(e_f1)

    # 6. Record this iteration's results for later saving to CSV
    def _cm_parts(cm):
        if cm is None or cm.shape != (2, 2):
            return {'TN': None, 'FP': None, 'FN': None, 'TP': None}
        tn, fp, fn, tp = cm.ravel()
        return {'TN': int(tn), 'FP': int(fp), 'FN': int(fn), 'TP': int(tp)}

    per_iteration_records.append({
        'iteration': i + 1, 'split': 'internal_test',
        'acc': acc, 'auc': auc, 'prec': prec, 'rec': rec, 'spec': spec, 'f1': f1,
        **_cm_parts(cm_internal),
    })
    if cm_external is not None:
        per_iteration_records.append({
            'iteration': i + 1, 'split': 'external_test',
            'acc': e_acc, 'auc': e_auc, 'prec': e_prec, 'rec': e_rec, 'spec': e_spec, 'f1': e_f1,
            **_cm_parts(cm_external),
        })

    # 7. Track the best iteration (by internal test AUC)
    if auc > best_test_auc:
        best_test_auc = auc
        best_model_state = {k: v.cpu() for k, v in trained_model_iter.state_dict().items()}
        best_iter_num = i + 1
        best_cm_internal = cm_internal
        best_cm_external = cm_external

    # 8. Mark this iteration as fully complete in the checkpoint
    save_checkpoint(i + 1, 0, None, None, float('inf'), 0, None)

# All iterations complete -- remove the checkpoint (no longer needed).
if os.path.exists(config.CHECKPOINT_PATH):
    os.remove(config.CHECKPOINT_PATH)
print("\n All iterations complete.")


In [ ]:
# %% Save Best Model + Confusion Matrices
model_save_path = os.path.join(config.MODEL_OUTPUT_DIR, config.MODEL_NAME)
torch.save({
    'model_state_dict': best_model_state,
    'config': {**{k: v for k, v in vars(Config).items() if not k.startswith('_')}, **vars(config)},
    'iteration': best_iter_num,
    'test_auc': best_test_auc,
}, model_save_path)
print(f" Saved best model (iteration {best_iter_num}, test AUC={best_test_auc:.4f}) to: {model_save_path}")


def plot_and_save_confusion_matrix(cm, title, save_path):
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No T2D', 'T2D'])
    fig, ax = plt.subplots(figsize=(5, 5))
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(save_path, dpi=200)
    plt.show()
    print(f"  Saved: {save_path}")


print("Saving confusion matrix plots for the best iteration...")
if best_cm_internal is not None:
    plot_and_save_confusion_matrix(
        best_cm_internal, f"Internal Test - Iter {best_iter_num} (best)",
        os.path.join(config.MODEL_OUTPUT_DIR, "confusion_matrix_internal_best.png")
    )
if best_cm_external is not None:
    plot_and_save_confusion_matrix(
        best_cm_external, f"External Test - Iter {best_iter_num} (best)",
        os.path.join(config.MODEL_OUTPUT_DIR, "confusion_matrix_external_best.png")
    )


In [ ]:
# %% Final Summary Statistics + Save All Metrics to File
print("\n" + "#" * 50)
print(f" MASK MODEL: FINAL RESULTS (OVER {NUM_ITERATIONS} ITERATIONS) ")
print("#" * 50)

print("\n--- INTERNAL TEST SET ---")
for metric, values in all_metrics.items():
    print(f"{metric.upper():10}: {np.mean(values):.4f} \u00b1 {np.std(values):.4f}")

print("\n--- EXTERNAL TEST SET ---")
for metric, values in external_metrics.items():
    if values:
        print(f"EXT_{metric.upper():6}: {np.mean(values):.4f} \u00b1 {np.std(values):.4f}")
    else:
        print(f"EXT_{metric.upper():6}: No data found.")
print("#" * 50)

# Per-iteration results (internal + external, one row each per iteration).
per_iter_df = pd.DataFrame(per_iteration_records)
per_iter_path = os.path.join(config.MODEL_OUTPUT_DIR, "metrics_per_iteration.csv")
per_iter_df.to_csv(per_iter_path, index=False)
print(f"\n Per-iteration metrics saved to: {per_iter_path}")

# Aggregate summary (mean +/- std per metric, both splits).
summary_rows = []
for metric, values in all_metrics.items():
    summary_rows.append({'split': 'internal_test', 'metric': metric, 'mean': np.mean(values), 'std': np.std(values)})
for metric, values in external_metrics.items():
    if values:
        summary_rows.append({'split': 'external_test', 'metric': metric, 'mean': np.mean(values), 'std': np.std(values)})
summary_df = pd.DataFrame(summary_rows)
summary_path = os.path.join(config.MODEL_OUTPUT_DIR, "metrics_summary.csv")
summary_df.to_csv(summary_path, index=False)
print(f" Summary (mean +/- std over {NUM_ITERATIONS} iterations) saved to: {summary_path}")

per_iter_df
